# 《Stochastic Interpolants: A Unifying Framework for Flows and Diffusions》完整学习笔记

**论文：** Michael S. Albergo、Nicholas M. Boffi、Eric Vanden-Eijnden  
**链接：** https://arxiv.org/abs/2303.08797

这次我们不再只讲前面的速度场、Score 和 ODE/SDE，而是把论文后半部分的理论也纳入，包括：

1. 随机插值的数学定义与概率密度。
2. 边缘速度场、连续性方程及其训练目标。
3. Score、Denoiser 以及不同预测参数化之间的关系。
4. 正向与反向 SDE、概率流 ODE。
5. ODE/SDE 的似然计算和 KL 误差控制。
6. 插值函数的设计、One-sided Interpolant 和 Mirror Interpolant。
7. Schrödinger Bridge 与最优传输的联系。
8. Reflow、生成模型之间的统一关系，以及论文实验。

**这次统一数学符号：**

- 用 $p_0(x)$、$p_1(x)$ 表示端点概率密度。
- 用 $p_{01}(x_0,x_1)$ 表示端点联合概率密度。
- 用 $p_t(x)$ 表示随机插值在时刻 $t$ 的边缘概率密度。
- 用 $b(t,x)$ 表示边缘速度场。
- 用 $s(t,x)=\nabla_x\log p_t(x)$ 表示 Score。
- 用 $X_t$ 表示人为构造的随机插值，$Y_t$ 表示通过 ODE 或 SDE 生成的新随机过程。
- 所有积分尽量使用普通的 $dx_0\,dx_1\,dz$ 形式。

对于没有普通联合密度的确定性耦合，我们会单独说明，不把测度积分强行写成不存在的密度。

整个笔记的核心是：**不仅给出论文结论，还要解释每个结论从哪个公式出发、经过什么变换、需要什么假设，以及最终有什么作用。**

## 第一部分：随机插值的基本理论（论文第 2.1～2.3 节）

### 1. 随机插值的定义

设两个端点随机变量为：

$$
X_0\sim p_0,\qquad X_1\sim p_1
$$

它们的联合分布由耦合决定。若存在联合密度，记为：

$$
p_{01}(x_0,x_1)
$$

另外引入独立的标准高斯变量：

$$
Z\sim\mathcal N(0,I_d)
$$

定义随机插值：

$$
\boxed{
X_t=I(t,X_0,X_1)+\gamma(t)Z
}
\tag{1.1}
$$

其中：

$$
I(0,x_0,x_1)=x_0,\qquad
I(1,x_0,x_1)=x_1
$$

$$
\gamma(0)=\gamma(1)=0
$$

因此：

$$
X_{t=0}=X_0,\qquad X_{t=1}=X_1
$$

也就是说，这个随机过程天然连接了两个端点分布。

### 2. 边缘概率密度

给定端点：

$$
X_0=x_0,\qquad X_1=x_1
$$

则：

$$
X_t=I(t,x_0,x_1)+\gamma(t)Z
$$

所以：

$$
X_t\mid x_0,x_1
\sim
\mathcal N(I(t,x_0,x_1),\gamma^2(t)I_d)
$$

对应条件概率密度：

$$
\boxed{
p_t(x\mid x_0,x_1)
=
\frac{
\exp\left[
-\frac{\|x-I(t,x_0,x_1)\|^2}{2\gamma^2(t)}
\right]
}{
(2\pi\gamma^2(t))^{d/2}
}
}
\tag{1.2}
$$

对所有端点积分：

$$
\boxed{
p_t(x)=
\int_{\mathbb R^d}\int_{\mathbb R^d}
p_t(x\mid x_0,x_1)
p_{01}(x_0,x_1)
\,dx_0\,dx_1
}
\tag{1.3}
$$

这是一个高斯混合密度。

注意这里没有对 $x$ 积分，因为我们正在求位置 $x$ 处的概率密度；积分掉的是未知端点 $x_0,x_1$。

### 3. 边缘速度场

对式 (1.1) 求时间导数：

$$
\dot X_t
=
\partial_tI(t,X_0,X_1)+\dot\gamma(t)Z
$$

定义：

$$
\boxed{
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
}
\tag{1.4}
$$

根据 Bayes 公式：

$$
p_t(x_0,x_1\mid x)
=
\frac{
p_t(x\mid x_0,x_1)p_{01}(x_0,x_1)
}{
p_t(x)
}
$$

又因为给定 $X_t=x$ 时：

$$
Z=\frac{x-I(t,X_0,X_1)}{\gamma(t)}
$$

所以：

$$
\boxed{
\begin{aligned}
b(t,x)
={}&
\int_{\mathbb R^d}\int_{\mathbb R^d}
\left[
\partial_tI(t,x_0,x_1)
+\frac{\dot\gamma(t)}{\gamma(t)}
(x-I(t,x_0,x_1))
\right]\\
&\qquad\cdot
p_t(x_0,x_1\mid x)
\,dx_0\,dx_1
\end{aligned}
}
\tag{1.5}
$$

这个公式告诉我们：

边缘速度场是所有可能经过位置 $x$ 的随机插值轨迹，其瞬时速度的条件平均。

### 4. 连续性方程

取光滑测试函数 $h(x)$，有：

$$
\mathbb E[h(X_t)]
=
\int_{\mathbb R^d}h(x)p_t(x)\,dx
$$

对时间求导：

$$
\frac{d}{dt}\mathbb E[h(X_t)]
=
\mathbb E[\nabla h(X_t)\cdot\dot X_t]
$$

利用条件期望：

$$
=
\mathbb E[\nabla h(X_t)\cdot b(t,X_t)]
$$

写成积分：

$$
\int h(x)\partial_tp_t(x)\,dx
=
\int\nabla h(x)\cdot b(t,x)p_t(x)\,dx
$$

右边分部积分：

$$
=
-\int h(x)\nabla\cdot[b(t,x)p_t(x)]\,dx
$$

由于 $h$ 任意：

$$
\boxed{
\partial_tp_t+\nabla\cdot(p_tb)=0
}
\tag{1.6}
$$

这就是连续性方程。

它描述的是概率质量守恒，而不是某个粒子的具体轨迹。

### 5. 为什么可以构造新的 ODE？

构造：

$$
\boxed{
\frac{dY_t}{dt}=b(t,Y_t),\qquad Y_0\sim p_0
}
\tag{1.7}
$$

记 $Y_t$ 的密度为 $q_t(x)$。

由 ODE 的概率输运性质：

$$
\partial_tq_t+\nabla\cdot(q_tb)=0
$$

而随机插值的密度满足：

$$
\partial_tp_t+\nabla\cdot(p_tb)=0
$$

并且：

$$
q_0=p_0
$$

如果这个初值问题的解唯一：

$$
\boxed{q_t(x)=p_t(x)}
\tag{1.8}
$$

所以随机插值和 ODE 具有相同的每时刻边缘分布。

但它们不是同一个随机过程。

### 6. Score 的推导

定义：

$$
s(t,x)=\nabla_x\log p_t(x)
$$

从式 (1.3) 出发，对 $x$ 求导：

$$
\nabla_xp_t(x)
=
\iint
\nabla_xp_t(x\mid x_0,x_1)
p_{01}(x_0,x_1)
\,dx_0\,dx_1
$$

条件密度为高斯，所以：

$$
\nabla_xp_t(x\mid x_0,x_1)
=
-\frac{x-I(t,x_0,x_1)}{\gamma^2(t)}
p_t(x\mid x_0,x_1)
$$

代入并除以 $p_t(x)$：

$$
s(t,x)
=
-\iint
\frac{x-I(t,x_0,x_1)}{\gamma^2(t)}
p_t(x_0,x_1\mid x)
\,dx_0\,dx_1
$$

利用：

$$
x-I(t,X_0,X_1)=\gamma(t)Z
$$

得到：

$$
\boxed{
s(t,x)
=
-\frac1{\gamma(t)}
\mathbb E[Z\mid X_t=x]
}
\tag{1.9}
$$

这个关系说明，预测高斯噪声的条件均值就可以恢复 Score。

### 7. 为什么 MSE 可以学习速度和 Score？

条件期望具有最小均方误差性质：

$$
\arg\min_f\mathbb E\|f(X)-Y\|^2
=
\mathbb E[Y\mid X]
$$

因此速度场可以通过：

$$
\boxed{
\mathcal L_b
=
\mathbb E\|
b_\theta(t,X_t)-\dot X_t
\|^2
}
\tag{1.10}
$$

学习。

噪声预测可以通过：

$$
\boxed{
\mathcal L_\eta
=
\mathbb E\|
\eta_\theta(t,X_t)-Z
\|^2
}
\tag{1.11}
$$

学习。

最优解分别为：

$$
b_\theta^*(t,x)=b(t,x)
$$

$$
\eta_\theta^*(t,x)=\mathbb E[Z\mid X_t=x]
$$

于是：

$$
s(t,x)=-\frac{\eta_\theta^*(t,x)}{\gamma(t)}
$$

这解释了为什么随机插值框架既能训练 Flow Matching 速度场，也能训练 Diffusion 所需的 Score。

---

## 第二部分：ODE、SDE 与概率密度演化

### 8. 从连续性方程推导 SDE 漂移

目标密度满足：

$$
\partial_tp_t=-\nabla\cdot(bp_t)
\tag{2.1}
$$

现在希望构造：

$$
dY_t=f(t,Y_t)dt+\sqrt{2\varepsilon(t)}dW_t
$$

使 $Y_t$ 的边缘密度也等于 $p_t$。

其 Fokker–Planck 方程为：

$$
\partial_tp_t
=
-\nabla\cdot(fp_t)+\varepsilon(t)\Delta p_t
\tag{2.2}
$$

将两式相等：

$$
-\nabla\cdot(bp_t)
=
-\nabla\cdot(fp_t)+\varepsilon\Delta p_t
$$

整理：

$$
\nabla\cdot[
(f-b)p_t-\varepsilon\nabla p_t
]=0
$$

为了构造一个满足条件的漂移场，选择括号内的概率流量差为零：

$$
(f-b)p_t-\varepsilon\nabla p_t=0
$$

于是：

$$
f=b+\varepsilon\frac{\nabla p_t}{p_t}
$$

根据 Score 定义：

$$
\boxed{
f(t,x)=b(t,x)+\varepsilon(t)s(t,x)
}
\tag{2.3}
$$

因此：

$$
\boxed{
dY_t=
[b(t,Y_t)+\varepsilon(t)s(t,Y_t)]dt
+
\sqrt{2\varepsilon(t)}dW_t
}
\tag{2.4}
$$

在相同初始分布及适当唯一性条件下，它与随机插值具有相同的边缘密度。

这个漂移场是一个方便的构造，不是所有满足要求的漂移场中的唯一选择。

### 9. 正向与反向 SDE 的推导

先固定时间约定：正向沿原始时间 $t:0\to1$，从 $p_0$ 到 $p_1$；反向沿原始时间 $t:1\to0$，从 $p_1$ 到 $p_0$。二者要复现的都是式 (2.1) 中的同一族边缘密度 $p_t$，但反向方程不能当作普通的正向 Itô SDE 来积分。

**正向：用概率流量匹配。** 对扩散系数只依赖时间的正向 SDE，式 (2.2) 可以写成

$$
\partial_t p_t=-\nabla\cdot J_F,\qquad
J_F=f_Fp_t-\varepsilon(t)\nabla p_t.
$$

目标连续性方程 (2.1) 的流量为 $J=bp_t$。取 $J_F=J$，并使用 $\nabla p_t=p_ts$，得到

$$
(f_F-b)p_t=\varepsilon(t)\nabla p_t
\quad\Longrightarrow\quad
\boxed{b_F(t,x):=f_F(t,x)=b(t,x)+\varepsilon(t)s(t,x)}.
$$

因此，从 $Y_0^F\sim p_0$ 出发，正向 SDE 是

$$
\boxed{
dY_t^F=b_F(t,Y_t^F)dt+\sqrt{2\varepsilon(t)}dW_t,
\qquad t:0\to1.
}
\tag{2.5}
$$

这与第 8 节的式 (2.3)–(2.4) 一致。这里把流量直接取为 $bp_t$ 是一个充分构造；仅要求散度相等，并不能唯一确定漂移。

**反向：先换成递增的时间。** 设 $\tau=1-t$、$q_\tau(x)=p_{1-\tau}(x)$，则 $q_0=p_1$。由式 (2.1) 和链式法则，

$$
\partial_\tau q_\tau
=-\partial_t p_t\big|_{t=1-\tau}
=\nabla\cdot\bigl[b(1-\tau,\cdot)q_\tau\bigr].
$$

在递增时间 $\tau$ 下，构造普通的正向 Itô SDE：

$$
dZ_\tau=g(\tau,Z_\tau)d\tau
+\sqrt{2\varepsilon(1-\tau)}d\widetilde W_\tau,
\qquad Z_0\sim p_1.
$$

它的 Fokker–Planck 方程是

$$
\partial_\tau q_\tau
=-\nabla\cdot(gq_\tau)+\varepsilon(1-\tau)\Delta q_\tau.
$$

令这个方程与上面的目标方程相同，并匹配流量：

$$
\underbrace{gq_\tau-\varepsilon(1-\tau)\nabla q_\tau}_{\text{$\tau$ 时间下的 SDE 流量}}
=\underbrace{-b(1-\tau,\cdot)q_\tau}_{\text{目标流量}}.
$$

因为 $\nabla\log q_\tau=s(1-\tau,\cdot)$，所以

$$
\boxed{
g(\tau,x)=-b(1-\tau,x)+\varepsilon(1-\tau)s(1-\tau,x).
}
$$

也就是说，实际按递增反向时间模拟时，用的是

$$
\boxed{
dZ_\tau=
[-b(1-\tau,Z_\tau)+\varepsilon(1-\tau)s(1-\tau,Z_\tau)]d\tau
+\sqrt{2\varepsilon(1-\tau)}d\widetilde W_\tau.
}
\tag{2.8}
$$

回到递减的原始时间 $t$，有 $d\tau=-dt$，故原始时间记号下的漂移是 $b_B(t,x)=-g(1-t,x)$：

$$
\boxed{b_B(t,x)=b(t,x)-\varepsilon(t)s(t,x)}.
$$

于是论文使用的反向 Itô 记号为

$$
\boxed{
dY_t^B=b_B(t,Y_t^B)dt+\sqrt{2\varepsilon(t)}d\overleftarrow W_t,
\qquad t:1\to0,\quad Y_1^B\sim p_1.
}
\tag{2.6}
$$

$d\overleftarrow W_t$ 表示按反向时间定义的布朗增量；它不能作为正向时间的 $dW_t$ 直接解释。等价地，反向过程的边缘密度满足论文的**反向 Fokker–Planck 方程**

$$
\partial_t p_t=-\nabla\cdot(b_Bp_t)-\varepsilon(t)\Delta p_t,
\qquad p_{t=1}=p_1.
$$

代入 $b_B=b-\varepsilon s$ 并用 $p_ts=\nabla p_t$，右侧正好化为 $-\nabla\cdot(bp_t)$。注意这里的扩散项在原始时间方程中带负号，是因为该方程从 $t=1$ 向 $t=0$ 求解；在递增时间 $\tau$ 的方程中，它仍是正号。

**与标准时间反转公式核对。** 在扩散系数与位置无关且密度足够光滑的条件下，正向 SDE 的时间反转漂移满足

$$
g(\tau,x)=-b_F(1-\tau,x)+2\varepsilon(1-\tau)s(1-\tau,x).
$$

代入 $b_F=b+\varepsilon s$，得到 $g=-b+\varepsilon s$，与流量推导一致。因此，在原始时间 $t$ 的漂移记号下，

$$
\boxed{b_F-b_B=2\varepsilon s,\qquad
b=\frac{b_F+b_B}{2}.}
\tag{2.7}
$$

这里的 $2\varepsilon s$ 是两个 SDE 漂移之差；从 ODE 流量构造单个正向或反向漂移时，各自只修正 $\varepsilon s$。

例如对不随时间变化的 $p_t=\mathcal N(0,1)$，取 $b=0$、$s=-x$、$\varepsilon=1$，就有 $b_F=-x$、$b_B=x$。正向过程 $dY_t^F=-Y_t^Fdt+\sqrt2dW_t$ 是平稳 Ornstein–Uhlenbeck 过程；反向原始时间写成 $dY_t^B=Y_t^Bdt+\sqrt2d\overleftarrow W_t$，换成递增的 $\tau$ 后仍是 $dZ_\tau=-Z_\tau d\tau+\sqrt2d\widetilde W_\tau$。这说明比较漂移符号时必须同时说明积分方向。

在边缘密度足够光滑、端点分布和方程解适当存在且唯一时，随机插值 $X_t$、概率流 ODE、正向 SDE 与反向 SDE 在每个时刻都具有 $p_t$，但它们的样本路径和端点耦合可以不同。若 $p_0$ 是噪声、$p_1$ 是数据，从噪声生成数据应沿 $t:0\to1$ 使用正向 SDE；此处的反向 SDE 从 $p_1$ 出发。推导对应论文第 2.2–2.3 节的正反向 Fokker–Planck 方程与 SDE：[原论文](https://arxiv.org/html/2303.08797)。

---

## 第三部分：似然与 KL 误差控制（论文第 2.4 节）

这是论文后半部分第一个非常重要的理论贡献。

它研究的是：

**如果神经网络没有精确学习到速度场和 Score，最终生成分布会偏离目标分布多少？**

### 10. ODE 的 KL 散度为什么难控制？

设真实速度为 $b$，网络预测速度为 $\hat b$。

真实密度满足：

$$
\partial_tp=-\nabla\cdot(bp)
$$

模型密度满足：

$$
\partial_tq=-\nabla\cdot(\hat bq)
$$

初始条件：

$$
p_0=q_0
$$

定义 KL 散度：

$$
\boxed{
D(t)=
\int_{\mathbb R^d}
p_t(x)\log\frac{p_t(x)}{q_t(x)}\,dx
}
\tag{3.1}
$$

对时间求导：

$$
\begin{aligned}
\frac{dD}{dt}
={}&
\int\partial_tp_t
\log\frac{p_t}{q_t}\,dx\\
&+
\int p_t
\left(
\frac{\partial_tp_t}{p_t}
-
\frac{\partial_tq_t}{q_t}
\right)dx
\end{aligned}
$$

由于：

$$
\int\partial_tp_t\,dx=0
$$

所以：

$$
\frac{dD}{dt}
=
\int\partial_tp_t\log\frac{p_t}{q_t}\,dx
-
\int\frac{p_t}{q_t}\partial_tq_t\,dx
\tag{3.2}
$$

代入两个连续性方程：

$$
\begin{aligned}
\frac{dD}{dt}
={}&
-\int\nabla\cdot(bp_t)
\log\frac{p_t}{q_t}\,dx\\
&+
\int\frac{p_t}{q_t}
\nabla\cdot(\hat bq_t)\,dx
\end{aligned}
$$

分别分部积分。

第一项：

$$
-\int\nabla\cdot(bp_t)\log\frac{p_t}{q_t}\,dx
=
\int bp_t\cdot\nabla\log\frac{p_t}{q_t}\,dx
$$

第二项：

$$
\begin{aligned}
\int\frac{p_t}{q_t}\nabla\cdot(\hat bq_t)\,dx
&=
-\int\hat bq_t\cdot\nabla\left(\frac{p_t}{q_t}\right)dx\\
&=
-\int\hat bp_t\cdot\nabla\log\frac{p_t}{q_t}\,dx
\end{aligned}
$$

合并：

$$
\boxed{
\frac{dD}{dt}
=
\int_{\mathbb R^d}
p_t(x)(b-\hat b)
\cdot
\nabla\log\frac{p_t}{q_t}
\,dx
}
\tag{3.3}
$$

定义真实与模型的 Score：

$$
s_p=\nabla\log p_t,\qquad
s_q=\nabla\log q_t
$$

于是：

$$
\nabla\log\frac{p_t}{q_t}=s_p-s_q
$$

所以：

$$
\boxed{
\frac{dD}{dt}
=
\int p_t(\hat b-b)\cdot(s_q-s_p)\,dx
}
\tag{3.4}
$$

由于初始 KL 为零：

$$
D(0)=0
$$

因此：

$$
\boxed{
D_{\mathrm{KL}}(p_1\|q_1)
=
\int_0^1\int_{\mathbb R^d}
p_t(\hat b-b)\cdot(s_q-s_p)
\,dx\,dt
}
\tag{3.5}
$$

这正是论文 Lemma 21 的结果。[arXiv](https://arxiv.org/html/2303.08797?utm_source=chatgpt.com)

### 11. 为什么速度 MSE 小，不一定意味着 KL 小？

对式 (3.5) 使用 Cauchy–Schwarz 不等式：

$$
D_{\mathrm{KL}}(p_1\|q_1)
\le
\left(
\int_0^1\int p_t\|\hat b-b\|^2dxdt
\right)^{1/2}
\left(
\int_0^1\int p_t\|s_q-s_p\|^2dxdt
\right)^{1/2}
\tag{3.6}
$$

第一项是速度误差。

第二项是 Fisher divergence：

$$
\boxed{
\mathrm{FI}(p_t\|q_t)
=
\int_{\mathbb R^d}
p_t(x)\|s_p(t,x)-s_q(t,x)\|^2dx
}
\tag{3.7}
$$

因此：

**仅仅让速度场 MSE 很小，不足以通过这个不等式直接保证最终 KL 很小。**

还需要控制模型分布和真实分布之间的 Score 差异。

这并不是说 ODE 一定比 SDE 差，而是说两者的误差控制机制不同。

### 12. 为什么 SDE 能得到更直接的 KL 上界？

考虑两个具有相同扩散强度的 SDE：

$$
dX_t=f(t,X_t)dt+\sqrt{2\varepsilon}dW_t
$$

$$
dY_t=\hat f(t,Y_t)dt+\sqrt{2\varepsilon}dW_t
$$

假设：

$$
X_0,Y_0\sim p_0
$$

其中 $\varepsilon>0$ 为常数。

令两个过程的路径分布分别为：

$$
P,\qquad Q
$$

在满足 Girsanov 定理的条件下：

$$
\boxed{
D_{\mathrm{KL}}(P\|Q)
=
\frac1{4\varepsilon}
\int_0^1
\mathbb E_{X_t\sim p_t}
\|f(t,X_t)-\hat f(t,X_t)\|^2dt
}
\tag{3.8}
$$

这里解释一下为什么系数是：

$$
\frac1{4\varepsilon}
$$

一般 Girsanov 公式中的漂移误差会乘以扩散矩阵的逆：

$$
\frac12
\left\|
\sigma^{-1}(f-\hat f)
\right\|^2
$$

由于：

$$
\sigma=\sqrt{2\varepsilon}I
$$

所以：

$$
\sigma^{-1}
=
\frac1{\sqrt{2\varepsilon}}I
$$

因此：

$$
\frac12
\left\|
\sigma^{-1}(f-\hat f)
\right\|^2
=
\frac1{4\varepsilon}\|f-\hat f\|^2
$$

现在利用 KL 散度的数据处理不等式。

最终状态：

$$
X_1
$$

只是整条路径的一个函数，因此：

$$
D_{\mathrm{KL}}(p_1\|q_1)
\le
D_{\mathrm{KL}}(P\|Q)
$$

得到：

$$
\boxed{
D_{\mathrm{KL}}(p_1\|q_1)
\le
\frac1{4\varepsilon}
\int_0^1\int_{\mathbb R^d}
p_t(x)\|f(t,x)-\hat f(t,x)\|^2
\,dx\,dt
}
\tag{3.9}
$$

这就是 SDE 相比 ODE 更容易得到漂移误差控制的原因。

### 13. 如何选择扩散强度？

真实漂移：

$$
f=b+\varepsilon s
$$

模型漂移：

$$
\hat f=\hat b+\varepsilon\hat s
$$

因此：

$$
\hat f-f
=
(\hat b-b)+\varepsilon(\hat s-s)
$$

使用：

$$
\|a+c\|^2\le2\|a\|^2+2\|c\|^2
$$

得到：

$$
\|\hat f-f\|^2
\le
2\|\hat b-b\|^2
+
2\varepsilon^2\|\hat s-s\|^2
$$

代入 KL 上界：

$$
\boxed{
D_{\mathrm{KL}}(p_1\|q_1)
\le
\frac{A}{2\varepsilon}
+
\frac{\varepsilon B}{2}
}
\tag{3.10}
$$

其中：

$$
A=
\int_0^1\int p_t\|\hat b-b\|^2\,dx\,dt
$$

$$
B=
\int_0^1\int p_t\|\hat s-s\|^2\,dx\,dt
$$

现在对 $\varepsilon$ 求导：

$$
\frac{d}{d\varepsilon}
\left(
\frac{A}{2\varepsilon}
+
\frac{\varepsilon B}{2}
\right)
=
-\frac{A}{2\varepsilon^2}+\frac B2
$$

令导数为零：

$$
-\frac{A}{2\varepsilon^2}+\frac B2=0
$$

得到：

$$
\boxed{
\varepsilon^*=\sqrt{\frac AB}
}
\tag{3.11}
$$

这说明理论上可以根据速度场和 Score 的估计精度来选择扩散强度。

但它优化的是理论上界，不保证有限步数采样时的实际最优扩散强度。

## 第四部分：如何计算生成模型的概率密度与似然？（论文第 2.5 节）

前面讨论的是训练误差与 KL 的关系。论文接下来研究另一个问题：

> 如果模型已经训练完成，给定一个真实样本 $x_1$，如何计算模型在这个位置的概率密度 $\hat p_1(x_1)$？

这与采样是两个不同任务。

采样只需要得到：

$$
Y_1\sim\hat p_1
$$

似然计算则需要求出：

$$
\log\hat p_1(x_1)
$$

### 14. ODE 的连续变量替换公式

考虑 ODE：

$$
\frac{dY_t}{dt}=b_\theta(t,Y_t)
\tag{4.1}
$$

其密度记为：

$$
q_t(x)
$$

满足：

$$
\partial_tq_t+\nabla\cdot(q_tb_\theta)=0
$$

将散度展开：

$$
\nabla\cdot(q_tb_\theta)
=
b_\theta\cdot\nabla q_t
+
q_t\nabla\cdot b_\theta
$$

所以：

$$
\partial_tq_t
+
b_\theta\cdot\nabla q_t
=
-q_t\nabla\cdot b_\theta
\tag{4.2}
$$

现在沿着 ODE 轨迹 $Y_t$ 求密度的全导数：

$$
\frac{d}{dt}q_t(Y_t)
=
\partial_tq_t(Y_t)
+
\nabla q_t(Y_t)\cdot\frac{dY_t}{dt}
$$

由于：

$$
\frac{dY_t}{dt}=b_\theta(t,Y_t)
$$

得到：

$$
\frac{d}{dt}q_t(Y_t)
=
\partial_tq_t(Y_t)
+
\nabla q_t(Y_t)\cdot b_\theta(t,Y_t)
$$

根据式 (4.2)：

$$
\frac{d}{dt}q_t(Y_t)
=
-q_t(Y_t)\nabla\cdot b_\theta(t,Y_t)
$$

两边除以 $q_t(Y_t)$：

$$
\boxed{
\frac{d}{dt}\log q_t(Y_t)
=
-\nabla\cdot b_\theta(t,Y_t)
}
\tag{4.3}
$$

对时间积分：

$$
\log q_1(Y_1)-\log q_0(Y_0)
=
-\int_0^1\nabla\cdot b_\theta(t,Y_t)\,dt
$$

因此：

$$
\boxed{
\log q_1(Y_1)
=
\log q_0(Y_0)
-
\int_0^1\nabla\cdot b_\theta(t,Y_t)\,dt
}
\tag{4.4}
$$

这就是 Continuous Normalizing Flow 的似然计算公式。

如果给定的是目标样本：

$$
Y_1=x_1
$$

那么先反向求解 ODE：

$$
x_1\rightarrow Y_0
$$

然后计算：

$$
\boxed{
\log q_1(x_1)
=
\log q_0(Y_0)
-
\int_0^1\nabla\cdot b_\theta(t,Y_t)\,dt
}
\tag{4.5}
$$

其中积分沿着连接 $Y_0$ 与 $x_1$ 的同一条 ODE 轨迹计算。

如果 $q_0$ 是标准高斯：

$$
\log q_0(Y_0)
=
-\frac d2\log(2\pi)-\frac12\|Y_0\|^2
$$

因此，ODE 可以给出模型密度的确定性计算公式，数值误差主要来自 ODE 积分和散度计算。

### 15. Cross-Entropy 如何计算？

给定真实数据分布 $p_1$，模型分布 $q_1$，交叉熵为：

$$
\boxed{
H(p_1,q_1)
=
-\int_{\mathbb R^d}p_1(x)\log q_1(x)\,dx
}
\tag{4.6}
$$

将式 (4.5) 代入：

$$
\begin{aligned}
H(p_1,q_1)
={}&
-\mathbb E_{x_1\sim p_1}
[\log q_0(Y_0)]\\
&+
\mathbb E_{x_1\sim p_1}
\left[
\int_0^1\nabla\cdot b_\theta(t,Y_t)\,dt
\right]
\end{aligned}
\tag{4.7}
$$

所以计算流程为：

给定真实数据 $x_1$，反向积分到 $Y_0$，再计算初始密度和沿途散度。

这也是为什么 ODE 模型在密度估计方面具有优势。

### 16. SDE 的密度为什么更难计算？

对于 SDE：

$$
dY_t=f(t,Y_t)dt+\sqrt{2\varepsilon}dW_t
$$

即使固定初始位置 $Y_0=y_0$，最终位置仍然是随机变量。

因此，不能像 ODE 那样，通过一个确定性的可逆映射计算 Jacobian。

论文给出的办法是引入**辅助随机过程**，利用 Fokker–Planck 方程的随机表示来计算密度。

下面推导其中最关键的结构。

考虑模型正向 Fokker–Planck 方程：

$$
\partial_tq
=
-\nabla\cdot(fq)+\varepsilon\Delta q
\tag{4.8}
$$

展开散度：

$$
\partial_tq
=
-f\cdot\nabla q
-
q\nabla\cdot f
+
\varepsilon\Delta q
$$

移项：

$$
\boxed{
\partial_tq
+
f\cdot\nabla q
-
\varepsilon\Delta q
=
-q\nabla\cdot f
}
\tag{4.9}
$$

这不是普通 ODE 沿特征线的形式，因为还存在：

$$
-\varepsilon\Delta q
$$

论文因此引入反向辅助扩散过程，使其生成元中的二阶导数与上式相匹配。

对于给定终点 $x$，沿辅助反向随机路径 $R_t$ 计算：

$$
\exp\left(
-\int_0^1\nabla\cdot f(t,R_t)\,dt
\right)q_0(R_0)
$$

然后对辅助随机路径取期望。

得到：

$$
\boxed{
q_1(x)
=
\mathbb E_{\mathrm{aux}}
\left[
q_0(R_0)
\exp\left(
-\int_0^1\nabla\cdot f(t,R_t)\,dt
\right)
\;\middle|\;
R_1=x
\right]
}
\tag{4.10}
$$

这里的辅助过程不是原来的正向生成 SDE；在论文的构造中，它使用相应的反向辅助动力学。

这一表达可以理解为 Feynman–Kac 型随机表示。

它与 ODE 的公式具有相似结构：

$$
\text{初始密度}
\times
\exp(-\text{沿路径积分的散度})
$$

但区别是：

- ODE 只有一条确定性路径。
- SDE 需要对一族随机路径取期望。

因此，SDE 的密度估计通常需要 Monte Carlo 近似。

**这里还有一个容易忽略的问题：**

即使模型训练了 Score 网络：

$$
s_\theta(t,x)
$$

它也不一定等于模型实际生成密度的：

$$
\nabla\log q_t(x)
$$

所以不能在模型存在误差时，直接把训练得到的 Score 当成真实模型密度的梯度。

论文在第 2.5 节明确讨论了这个限制。[arXiv](https://arxiv.org/html/2303.08797?utm_source=chatgpt.com)

---

## 第五部分：Diffusive Interpolants 与 Brownian Bridge（论文第 3.1 节）

这一节开始研究随机插值与真正随机扩散桥之间的关系。

### 17. Brownian Bridge 是什么？

先考虑标准布朗运动：

$$
W_t
$$

满足：

$$
W_0=0
$$

$$
W_t-W_s\sim\mathcal N(0,(t-s)I)
$$

现在希望构造一个随机过程：

$$
B_t
$$

使它同时满足：

$$
B_0=0,\qquad B_1=0
$$

这就是标准 Brownian Bridge。

它可以写成：

$$
\boxed{
B_t=W_t-tW_1
}
\tag{5.1}
$$

验证端点：

当 $t=0$：

$$
B_0=W_0=0
$$

当 $t=1$：

$$
B_1=W_1-W_1=0
$$

### 18. Brownian Bridge 的方差为什么是 $t(1-t)$？

由：

$$
B_t=W_t-tW_1
$$

计算方差：

$$
\begin{aligned}
\operatorname{Var}(B_t)
={}&
\operatorname{Var}(W_t)
+t^2\operatorname{Var}(W_1)\\
&-2t\operatorname{Cov}(W_t,W_1)
\end{aligned}
$$

因为：

$$
\operatorname{Var}(W_t)=t
$$

$$
\operatorname{Var}(W_1)=1
$$

$$
\operatorname{Cov}(W_t,W_1)=t
$$

所以：

$$
\begin{aligned}
\operatorname{Var}(B_t)
&=t+t^2-2t^2\\
&=t-t^2
\end{aligned}
$$

因此：

$$
\boxed{
B_t\sim\mathcal N(0,t(1-t)I)
}
\tag{5.2}
$$

对于固定时刻 $t$，可以写成分布相等：

$$
\boxed{
B_t\overset d=\sqrt{t(1-t)}Z
}
\tag{5.3}
$$

其中：

$$
Z\sim\mathcal N(0,I)
$$

注意，这只是单时刻分布相等。

不能据此认为：

$$
B_t=\sqrt{t(1-t)}Z
$$

作为整个随机过程成立。

前者是由布朗运动构造的连续但不可微过程，后者在固定 $Z$ 后是一条确定性曲线。

### 19. 为什么随机插值与 Brownian Bridge 可以具有相同的边缘分布？

论文考虑：

$$
\boxed{
X_t^{\mathrm{diff}}
=
I(t,X_0,X_1)+\sqrt{2a}\,B_t
}
\tag{5.4}
$$

其中：

$$
a>0
$$

根据式 (5.3)，对每个固定 $t$：

$$
\sqrt{2a}B_t
\overset d=
\sqrt{2at(1-t)}Z
$$

因此：

$$
X_t^{\mathrm{diff}}
\overset d=
I(t,X_0,X_1)+\sqrt{2at(1-t)}Z
$$

也就是说，它与随机插值：

$$
\boxed{
X_t=
I(t,X_0,X_1)+\gamma(t)Z
}
$$

在选择：

$$
\boxed{
\gamma(t)=\sqrt{2at(1-t)}
}
\tag{5.5}
$$

时，具有相同的单时刻边缘分布。

这就是论文第 3.1 节的重要结论。

它说明：

**可以用一个固定高斯随机变量构造随机插值，替代复杂 Brownian Bridge 的单时刻分布计算。**

这样训练时不需要模拟整条布朗路径。

### 20. Brownian Bridge 对应的 SDE

Brownian Bridge 还满足：

$$
\boxed{
dB_t=-\frac{B_t}{1-t}dt+dW_t
}
\tag{5.6}
$$

为什么会有：

$$
-\frac{B_t}{1-t}
$$

这一项？

因为 Brownian Bridge 被要求在 $t=1$ 回到零。

当 $t$ 接近 1 时，如果当前位置：

$$
B_t\neq0
$$

那么漂移：

$$
-\frac{B_t}{1-t}
$$

会迅速增大，将过程拉回原点。

例如：

$$
B_t>0
$$

则漂移为负，推动它向零运动。

当：

$$
B_t<0
$$

则漂移为正，同样推动它向零运动。

这就是 Brownian Bridge 的回归端点机制。

### 21. 与 Doob's $h$-transform 的关系

Doob's $h$-transform 可以用于构造满足特定终点条件的扩散过程。

考虑原始 SDE：

$$
dX_t=f(t,X_t)dt+\sqrt{2\varepsilon}dW_t
$$

希望条件化到：

$$
X_1=y
$$

设：

$$
h(t,x)=p(X_1=y\mid X_t=x)
$$

这里 $h$ 是终点条件对应的转移密度函数。

在适当条件下，条件化过程的漂移变为：

$$
\boxed{
f_{\mathrm{bridge}}(t,x)
=
f(t,x)+2\varepsilon\nabla_x\log h(t,x)
}
\tag{5.7}
$$

这个公式说明：

为了让扩散过程最终到达指定位置，需要在原漂移上增加一个与终点条件概率有关的修正项。

对于 Brownian Bridge：

$$
f=0,\qquad \varepsilon=\frac12
$$

从时刻 $t$ 的位置 $x$ 到终点零的转移密度为：

$$
h(t,x)
\propto
\exp\left(-\frac{\|x\|^2}{2(1-t)}\right)
$$

所以：

$$
\nabla_x\log h(t,x)
=
-\frac{x}{1-t}
$$

代入：

$$
f_{\mathrm{bridge}}
=
0+2\cdot\frac12
\left(-\frac{x}{1-t}\right)
$$

得到：

$$
\boxed{
f_{\mathrm{bridge}}(t,x)
=
-\frac{x}{1-t}
}
$$

这就恢复了 Brownian Bridge 的漂移。

### 22. 随机插值相比直接使用扩散桥有什么优势？

直接构造一般扩散桥时，需要知道：

$$
h(t,x)
$$

或者对应的条件转移概率。

对于复杂数据分布，这通常很难计算。

随机插值则先规定：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

然后通过条件期望学习速度场和 Score。

这样就绕开了显式求解复杂 Doob $h$-transform 的需求。

论文还讨论了从固定起点生成任意目标分布的条件扩散构造，这一点是普通光滑 ODE 无法直接实现的：固定一个确定性初始点，在唯一解 ODE 下只能得到一个确定性终点，而具有非零扩散的 SDE 可以产生非退化的终点分布。[arXiv](https://arxiv.org/html/2303.08797?utm_source=chatgpt.com)

## 第六部分：One-sided Interpolant 与 Mirror Interpolant（论文第 3.2～3.3 节）

### 23. 为什么初始分布是高斯时可以简化？

考虑一般线性随机插值：

$$
X_t=\alpha(t)X_0+\beta(t)X_1+\gamma(t)Z
\tag{6.1}
$$

假设：

$$
X_0,Z\sim\mathcal N(0,I)
$$

并且二者与 $X_1$ 独立。

那么：

$$
\alpha(t)X_0+\gamma(t)Z
$$

仍然是高斯变量。

均值：

$$
\mathbb E[\alpha X_0+\gamma Z]=0
$$

协方差：

$$
\begin{aligned}
\operatorname{Cov}(\alpha X_0+\gamma Z)
&=
\alpha^2\operatorname{Cov}(X_0)
+\gamma^2\operatorname{Cov}(Z)\\
&=
(\alpha^2+\gamma^2)I
\end{aligned}
$$

定义：

$$
\sigma(t)=\sqrt{\alpha^2(t)+\gamma^2(t)}
$$

因此：

$$
\alpha(t)X_0+\gamma(t)Z
\overset d=
\sigma(t)\widetilde Z
$$

其中：

$$
\widetilde Z\sim\mathcal N(0,I)
$$

于是每个固定时刻的分布可以用：

$$
\boxed{
X_t=\sigma(t)\widetilde Z+\beta(t)X_1
}
\tag{6.2}
$$

表示。

这就是 One-sided Interpolant 的核心思想。

**两个独立高斯随机变量可以合并为一个高斯随机变量。**

因此，对于高斯起点，不一定需要显式保留三个随机变量。

### 24. One-sided Interpolant 的一般定义

论文定义：

$$
\boxed{
X_t=\alpha(t)Z+J(t,X_1)
}
\tag{6.3}
$$

其中：

$$
Z\sim\mathcal N(0,I),\qquad Z\perp X_1
$$

要求：

$$
\alpha(0)=1,\qquad \alpha(1)=0
$$

$$
J(0,X_1)=0,\qquad J(1,X_1)=X_1
$$

于是：

$$
X_0=Z
$$

$$
X_1=X_1
$$

因此它从标准高斯分布连接到真实数据分布。

特别地，空间线性 One-sided Interpolant 为：

$$
\boxed{
X_t=\alpha(t)Z+\beta(t)X_1
}
\tag{6.4}
$$

其中：

$$
\beta(0)=0,\qquad\beta(1)=1
$$

### 25. 为什么只预测一个噪声就够了？

对式 (6.4) 求时间导数：

$$
\dot X_t=\dot\alpha(t)Z+\dot\beta(t)X_1
$$

定义：

$$
\eta_z(t,x)=\mathbb E[Z\mid X_t=x]
$$

$$
\eta_1(t,x)=\mathbb E[X_1\mid X_t=x]
$$

那么：

$$
\boxed{
b(t,x)
=
\dot\alpha(t)\eta_z(t,x)
+
\dot\beta(t)\eta_1(t,x)
}
\tag{6.5}
$$

另一方面：

$$
X_t=\alpha Z+\beta X_1
$$

在条件 $X_t=x$ 下取期望：

$$
x=\alpha\eta_z+\beta\eta_1
$$

所以：

$$
\boxed{
\eta_1(t,x)
=
\frac{x-\alpha(t)\eta_z(t,x)}{\beta(t)}
}
\tag{6.6}
$$

代入速度场：

$$
\begin{aligned}
b(t,x)
&=
\dot\alpha\eta_z
+
\dot\beta
\frac{x-\alpha\eta_z}{\beta}\\
&=
\frac{\dot\beta}{\beta}x
+
\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
\right)\eta_z
\end{aligned}
$$

最终：

$$
\boxed{
b(t,x)
=
\frac{\dot\beta(t)}{\beta(t)}x
+
\left[
\dot\alpha(t)
-
\frac{\alpha(t)\dot\beta(t)}{\beta(t)}
\right]\eta_z(t,x)
}
\tag{6.7}
$$

而 Score：

$$
\boxed{
s(t,x)=-\frac{\eta_z(t,x)}{\alpha(t)}
}
\tag{6.8}
$$

所以：

$$
\boxed{
\eta_z
\Longrightarrow
\begin{cases}
b\\
s
\end{cases}
}
$$

这就是论文中 One-sided Interpolant 的重要优势。

**只需要训练一个噪声预测网络，就能同时构造 ODE 和 SDE。**

但式 (6.7) 在 $\beta=0$ 的端点不能直接使用，需要通过原始条件期望表达式求极限。

### 26. Mirror Interpolant 是什么？

论文第 3.3 节还提出 Mirror Interpolant。

与普通生成模型不同，Mirror Interpolant 的两个端点分布相同。

设：

$$
X_1\sim p_{\mathrm{data}}
$$

定义：

$$
\boxed{
X_t=K(t,X_1)+\gamma(t)Z
}
\tag{6.9}
$$

要求：

$$
K(0,X_1)=X_1
$$

$$
K(1,X_1)=X_1
$$

以及：

$$
\gamma(0)=\gamma(1)=0
$$

所以：

$$
X_0=X_1,\qquad X_{t=1}=X_1
$$

也就是说：

$$
\boxed{
p_{t=0}=p_{t=1}=p_{\mathrm{data}}
}
\tag{6.10}
$$

中间时刻可以经过更简单的分布。

例如，选择：

$$
K(t,x_1)=0
$$

在某个中间时间区间成立，那么：

$$
X_t=\gamma(t)Z
$$

因此：

$$
X_t\sim\mathcal N(0,\gamma^2(t)I)
$$

此时模型可以先把真实数据分布变换为高斯，再从高斯返回真实数据。

这类似于将编码和解码两个过程连接起来。

但需要注意，Mirror Interpolant 不意味着每个时刻的分布都相同；它只要求两个端点分布相同。

### 27. Mirror Interpolant 为什么有意义？

通常生成模型从：

$$
p_0=\mathcal N(0,I)
$$

生成：

$$
p_1=p_{\mathrm{data}}
$$

而 Mirror Interpolant 允许从数据分布出发，经过一个中间参考分布，再回到数据分布。

它可以用于研究：

- 数据到潜在分布的编码过程。
- 潜在分布到数据的解码过程。
- 具有相同起点和终点分布的随机变换。

如果选择最简单的：

$$
K(t,X_1)=X_1
$$

那么：

$$
X_t=X_1+\gamma(t)Z
$$

它就是先给数据加噪，再逐渐去噪。

此时：

$$
\dot X_t=\dot\gamma(t)Z
$$

边缘速度：

$$
b(t,x)=\dot\gamma(t)\eta_z(t,x)
$$

Score：

$$
s(t,x)=-\frac{\eta_z(t,x)}{\gamma(t)}
$$

因此：

$$
\boxed{
b(t,x)=-\gamma(t)\dot\gamma(t)s(t,x)
}
\tag{6.11}
$$

这说明在这个特殊的 Mirror Interpolant 中：

**边缘速度完全由 Score 决定。**

---

## 第七部分：Schrödinger Bridge 的完整推导（论文第 3.4 节）

这是论文后半部分数学上最重要的扩展之一。

我们之前研究过最优传输，现在需要理解：

**如果不仅希望连接两个分布，还希望在随机扩散条件下找到某种最优的概率路径，该怎么做？**

### 28. 从最优传输开始

经典动态最优传输可以写成：

$$
\boxed{
\min_{p,v}
\int_0^1\int_{\mathbb R^d}
\frac12\|v(t,x)\|^2p_t(x)
\,dx\,dt
}
\tag{7.1}
$$

约束：

$$
\partial_tp+\nabla\cdot(pv)=0
$$

以及：

$$
p_{t=0}=p_0,\qquad p_{t=1}=p_1
$$

其中：

- $p_t(x)$：待优化的中间概率密度。
- $v(t,x)$：概率质量的输运速度。
- $\frac12\|v\|^2$：单位质量的运动能量。

它希望找到连接两个分布的低能量输运路径。

### 29. Schrödinger Bridge 增加了什么？

Schrödinger Bridge 考虑带扩散的输运：

$$
\boxed{
\partial_tp+\nabla\cdot(pu)
=
\varepsilon\Delta p
}
\tag{7.2}
$$

其中：

$$
\varepsilon>0
$$

表示扩散强度。

优化目标可以写成：

$$
\boxed{
\min_{p,u}
\int_0^1\int_{\mathbb R^d}
\frac12\|u(t,x)\|^2p_t(x)
\,dx\,dt
}
\tag{7.3}
$$

约束：

$$
\begin{cases}
\partial_tp+\nabla\cdot(pu)=\varepsilon\Delta p\\
p(0,x)=p_0(x)\\
p(1,x)=p_1(x)
\end{cases}
$$

这里采用了目标前面的 $1/2$ 约定，最优解不受这个常数因子的影响。

Schrödinger Bridge 可以理解为：

在固定扩散强度下，寻找一个能够连接两个端点分布，同时使漂移控制能量最小的随机过程。

### 30. 使用拉格朗日乘子推导最优漂移

引入拉格朗日乘子：

$$
\lambda(t,x)
$$

构造：

$$
\begin{aligned}
\mathcal J[p,u,\lambda]
={}&
\int_0^1\int_{\mathbb R^d}
\frac12\|u\|^2p\,dxdt\\
&+
\int_0^1\int_{\mathbb R^d}
\lambda
\left[
\partial_tp+\nabla\cdot(pu)-\varepsilon\Delta p
\right]dxdt
\end{aligned}
\tag{7.4}
$$

接下来对约束项分部积分。

时间项：

$$
\int\lambda\partial_tp\,dt
=
[\lambda p]_{t=0}^{t=1}
-
\int p\partial_t\lambda\,dt
$$

由于端点分布固定，端点项在变分时不影响内部最优条件。

空间散度项：

$$
\int\lambda\nabla\cdot(pu)\,dx
=
-\int p\,u\cdot\nabla\lambda\,dx
$$

扩散项：

$$
-\varepsilon\int\lambda\Delta p\,dx
=
-\varepsilon\int p\Delta\lambda\,dx
$$

因此，忽略固定端点的边界变分后：

$$
\boxed{
\mathcal J
=
\int_0^1\int_{\mathbb R^d}
p
\left[
\frac12\|u\|^2
-u\cdot\nabla\lambda
-\partial_t\lambda
-\varepsilon\Delta\lambda
\right]dxdt
+\text{边界项}
}
\tag{7.5}
$$

### 31. 对漂移 $u$ 求变分

只考虑与 $u$ 有关的部分：

$$
\frac12\|u\|^2-u\cdot\nabla\lambda
$$

对 $u$ 求导：

$$
u-\nabla\lambda
$$

令其为零：

$$
\boxed{
u^*(t,x)=\nabla\lambda(t,x)
}
\tag{7.6}
$$

这说明：

**Schrödinger Bridge 的最优漂移是一个梯度场。**

将：

$$
u=\nabla\lambda
$$

代回式 (7.5)：

$$
\frac12\|\nabla\lambda\|^2
-
\|\nabla\lambda\|^2
=
-\frac12\|\nabla\lambda\|^2
$$

因此内部被积函数变为：

$$
-p
\left[
\partial_t\lambda
+
\frac12\|\nabla\lambda\|^2
+
\varepsilon\Delta\lambda
\right]
$$

对 $p$ 求变分，得到：

$$
\boxed{
\partial_t\lambda
+
\frac12\|\nabla\lambda\|^2
+
\varepsilon\Delta\lambda
=
0
}
\tag{7.7}
$$

也就是：

$$
\boxed{
\partial_t\lambda
+
\frac12\|\nabla\lambda\|^2
=
-\varepsilon\Delta\lambda
}
$$

这就是一个带扩散项的 Hamilton–Jacobi 方程。

最终，Schrödinger Bridge 的最优条件为：

$$
\boxed{
\begin{cases}
\partial_tp+\nabla\cdot(p\nabla\lambda)
=
\varepsilon\Delta p\\[4pt]
\partial_t\lambda+\frac12\|\nabla\lambda\|^2
=
-\varepsilon\Delta\lambda
\end{cases}
}
\tag{7.8}
$$

并满足两个端点分布约束。

这对应论文第 3.4 节给出的 Euler–Lagrange 方程。

### 32. 随机插值如何求解 Schrödinger Bridge？

现在回到随机插值：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

它对应的正向 SDE 漂移：

$$
u=b+\varepsilon s
$$

其中：

$$
b(t,x)
=
\mathbb E[\partial_tI+\dot\gamma Z\mid X_t=x]
$$

$$
s(t,x)
=
-\frac1\gamma\mathbb E[Z\mid X_t=x]
$$

所以：

$$
\begin{aligned}
u(t,x)
&=
\mathbb E[\partial_tI+\dot\gamma Z\mid X_t=x]
-
\frac{\varepsilon}{\gamma}
\mathbb E[Z\mid X_t=x]\\
&=
\mathbb E\left[
\partial_tI+
\left(\dot\gamma-\frac{\varepsilon}{\gamma}\right)Z
\mid X_t=x
\right]
\end{aligned}
$$

因此：

$$
\boxed{
u(t,x)
=
\mathbb E\left[
\partial_tI+
\left(\dot\gamma-\frac{\varepsilon}{\gamma}\right)Z
\mid X_t=x
\right]
}
\tag{7.9}
$$

根据条件期望的 MSE 最优性，可以构造：

$$
\boxed{
\begin{aligned}
\mathcal L(I,\hat u)
=
\int_0^1\mathbb E\Big[
&\frac12\|\hat u(t,X_t)\|^2\\
&-
\hat u(t,X_t)\cdot
\left(
\partial_tI+
\left(\dot\gamma-\frac{\varepsilon}{\gamma}\right)Z
\right)
\Big]dt
\end{aligned}
}
\tag{7.10}
$$

对于固定 $I$，最小化：

$$
\min_{\hat u}\mathcal L(I,\hat u)
$$

得到：

$$
\hat u^*=u
$$

并且最小值为：

$$
\boxed{
\min_{\hat u}\mathcal L(I,\hat u)
=
-\frac12
\int_0^1\mathbb E\|u(t,X_t)\|^2dt
}
\tag{7.11}
$$

因此，如果进一步最大化 $I$：

$$
\boxed{
\max_I\min_{\hat u}\mathcal L(I,\hat u)
}
\tag{7.12}
$$

就相当于在随机插值能够表示的概率路径中，最小化正向漂移的运动能量。

这正好与 Schrödinger Bridge 的优化目标对应。

### 33. 为什么外层是最大化？

这一点容易混淆。

因为内层优化得到：

$$
\min_{\hat u}\mathcal L
=
-\frac12\mathbb E\|u\|^2
$$

我们希望：

$$
\mathbb E\|u\|^2
$$

尽可能小。

因此，需要让它前面的负号对应的目标尽可能大：

$$
\max_I
\left(
-\frac12\mathbb E\|u\|^2
\right)
$$

所以外层是：

$$
\max_I
$$

而不是：

$$
\min_I
$$

这就是 max-min 结构的来源。

### 34. 这个 max-min 是否一定得到真正的 Schrödinger Bridge？

这里需要区分两个层次。

**第一层：对于固定的随机插值 $I$，可以学习到它对应的正向 SDE 漂移。**

这是由条件期望最优性直接保证的。

**第二层：优化 $I$ 后，是否能够覆盖真正的 Schrödinger Bridge 最优概率路径？**

这需要额外假设。

论文 Theorem 41 依赖于一个重要条件：存在足够光滑的可逆映射，可以在标准高斯分布与最优中间密度之间建立对应关系。

在这个假设下，论文证明相应的随机插值函数类能够表示最优概率路径，进而建立 max-min 优化与 Schrödinger Bridge 的联系。

因此，不能简单理解为：

> 任意一个有限参数神经网络 $I_\theta$，只要训练 max-min，就一定得到全局最优 Schrödinger Bridge。

理论结论依赖函数类的表达能力、正则性以及全局优化条件。

论文也明确指出，利用该 max-min 方案数值求解 Schrödinger Bridge 的系统实验仍属于后续研究方向。([arxiv.org](https://arxiv.org/html/2303.08797))

---

## 第八部分：空间线性插值的参数化设计（论文第 4 节）

### 35. 为什么要研究空间线性插值？

一般随机插值为：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

其中 $I$ 可以非常复杂。

但实践中最常用的是：

$$
\boxed{
X_t=\alpha(t)X_0+\beta(t)X_1+\gamma(t)Z
}
\tag{8.1}
$$

其中：

$$
\alpha(0)=1,\qquad\alpha(1)=0
$$

$$
\beta(0)=0,\qquad\beta(1)=1
$$

$$
\gamma(0)=\gamma(1)=0
$$

这个形式能够覆盖大量常见生成模型。

### 36. 速度场的因子分解

对时间求导：

$$
\dot X_t
=
\dot\alpha X_0+\dot\beta X_1+\dot\gamma Z
$$

定义：

$$
\eta_0(t,x)=\mathbb E[X_0\mid X_t=x]
$$

$$
\eta_1(t,x)=\mathbb E[X_1\mid X_t=x]
$$

$$
\eta_z(t,x)=\mathbb E[Z\mid X_t=x]
$$

因此：

$$
\boxed{
b(t,x)
=
\dot\alpha(t)\eta_0(t,x)
+
\dot\beta(t)\eta_1(t,x)
+
\dot\gamma(t)\eta_z(t,x)
}
\tag{8.2}
$$

Score：

$$
\boxed{
s(t,x)=-\frac{\eta_z(t,x)}{\gamma(t)}
}
\tag{8.3}
$$

另一方面：

$$
X_t=\alpha X_0+\beta X_1+\gamma Z
$$

在给定 $X_t=x$ 后取期望：

$$
\boxed{
x=\alpha\eta_0+\beta\eta_1+\gamma\eta_z
}
\tag{8.4}
$$

这说明三个条件期望并不是完全独立的。

只要其中两个已知，就可以通过式 (8.4) 恢复第三个，前提是对应系数非零。

这给模型参数化提供了自由度。

例如可以训练：

$$
(\eta_0,\eta_1)
$$

也可以训练：

$$
(\eta_1,\eta_z)
$$

或者直接训练：

$$
b
$$

### 37. 为什么要考虑方差保持？

假设：

$$
\mathbb E[X_0]=\mathbb E[X_1]=0
$$

并且：

$$
\operatorname{Cov}(X_0)
=
\operatorname{Cov}(X_1)
=
I
$$

再假设三个随机变量相互独立。

那么：

$$
\operatorname{Cov}(X_t)
=
\alpha^2I+\beta^2I+\gamma^2I
$$

所以：

$$
\boxed{
\operatorname{Cov}(X_t)
=
(\alpha^2+\beta^2+\gamma^2)I
}
\tag{8.5}
$$

如果希望中间时刻始终保持单位协方差：

$$
\operatorname{Cov}(X_t)=I
$$

则需要：

$$
\boxed{
\alpha^2(t)+\beta^2(t)+\gamma^2(t)=1
}
\tag{8.6}
$$

这个条件有助于避免中间变量的整体尺度剧烈变化。

不过要注意：如果 $X_0,X_1$ 相关，那么协方差还会出现交叉项：

$$
\alpha\beta
\left[
\operatorname{Cov}(X_0,X_1)
+
\operatorname{Cov}(X_1,X_0)
\right]
$$

因此式 (8.6) 不能无条件用于任意耦合。

### 38. 线性插值

选择：

$$
\alpha(t)=1-t
$$

$$
\beta(t)=t
$$

为了满足：

$$
\alpha^2+\beta^2+\gamma^2=1
$$

需要：

$$
\gamma^2
=
1-(1-t)^2-t^2
$$

展开：

$$
\gamma^2
=
1-(1-2t+t^2)-t^2
$$

整理：

$$
\gamma^2=2t(1-t)
$$

所以：

$$
\boxed{
\gamma(t)=\sqrt{2t(1-t)}
}
\tag{8.7}
$$

最终：

$$
\boxed{
X_t=(1-t)X_0+tX_1+\sqrt{2t(1-t)}Z
}
\tag{8.8}
$$

这是论文中的一个典型随机插值。

### 39. 三角函数插值

另一种选择是：

$$
\alpha(t)=\cos\left(\frac{\pi t}{2}\right)
$$

$$
\beta(t)=\sin\left(\frac{\pi t}{2}\right)
$$

当：

$$
\gamma(t)=0
$$

时：

$$
\alpha^2+\beta^2=1
$$

因为：

$$
\cos^2\theta+\sin^2\theta=1
$$

所以：

$$
\boxed{
X_t=
\cos\left(\frac{\pi t}{2}\right)X_0
+
\sin\left(\frac{\pi t}{2}\right)X_1
}
\tag{8.9}
$$

这种插值在独立、单位协方差端点的情况下天然保持协方差尺度。

如果希望增加额外噪声，可以令：

$$
\alpha(t)
=
\sqrt{1-\gamma^2(t)}
\cos\left(\frac{\pi t}{2}\right)
$$

$$
\beta(t)
=
\sqrt{1-\gamma^2(t)}
\sin\left(\frac{\pi t}{2}\right)
$$

这样仍然满足：

$$
\alpha^2+\beta^2+\gamma^2=1
$$

### 40. Gaussian Encoding–Decoding 插值

论文还讨论了一类特殊的插值路径：

先逐渐减小原始数据端点的贡献，使中间状态接近一个简单高斯分布，然后再逐渐增加目标数据端点的贡献。

可以理解为：

$$
p_0
\rightarrow
\mathcal N(0,I)
\rightarrow
p_1
$$

与普通线性插值不同，这种路径在中间时刻可以显式经过一个简单参考分布。

它的优势是中间概率密度可能更容易学习。

代价是概率路径更长，ODE/SDE 的数值积分不一定更高效。

### 41. $\gamma(t)$ 与 $\varepsilon(t)$ 的区别

这两个参数一定要区分。

随机插值：

$$
X_t=I(t,X_0,X_1)+\gamma(t)Z
$$

其中：

$$
\gamma(t)
$$

控制中间概率路径的平滑程度。

SDE：

$$
dY_t=
[b(t,Y_t)+\varepsilon(t)s(t,Y_t)]dt
+
\sqrt{2\varepsilon(t)}dW_t
$$

其中：

$$
\varepsilon(t)
$$

控制生成过程的随机性。

所以：

$$
\boxed{
\gamma(t)\text{ 决定目标概率路径}
}
$$

$$
\boxed{
\varepsilon(t)\text{ 决定如何随机地实现这条路径}
}
$$

改变 $\gamma$ 通常改变 $p_t$。

改变 $\varepsilon$ 并同步使用正确的 Score 修正，则可以保持相同的 $p_t$。

论文第 4 节使用高斯混合分布展示了这两种参数对中间概率密度和采样轨迹的不同影响。([arxiv.org](https://arxiv.org/html/2303.08797))

## 第九部分：与 VP-SDE、Denoising 和 Rectified Flow 的联系（论文第 5 节）

### 42. VP-SDE 如何对应随机插值？

考虑 Ornstein–Uhlenbeck SDE：

$$
\boxed{
dY_\tau=-Y_\tau d\tau+\sqrt2dW_\tau
}
\tag{9.1}
$$

初始状态：

$$
Y_0=X_1
$$

其中 $X_1$ 是真实数据。

这个 SDE 的解析解为：

$$
\boxed{
Y_\tau
=
e^{-\tau}X_1
+
\sqrt2
\int_0^\tau e^{-(\tau-r)}dW_r
}
\tag{9.2}
$$

第二项是均值为零的高斯随机变量。

根据 Itô 等距：

$$
\begin{aligned}
\operatorname{Var}
\left(
\sqrt2\int_0^\tau e^{-(\tau-r)}dW_r
\right)
&=
2\int_0^\tau e^{-2(\tau-r)}dr\\
&=
1-e^{-2\tau}
\end{aligned}
$$

所以：

$$
\boxed{
Y_\tau\mid X_1=x_1
\sim
\mathcal N
\left(
e^{-\tau}x_1,
(1-e^{-2\tau})I
\right)
}
\tag{9.3}
$$

现在定义：

$$
t=e^{-\tau}
$$

则：

$$
e^{-2\tau}=t^2
$$

因此可以构造具有相同单时刻分布的随机插值：

$$
\boxed{
X_t=tX_1+\sqrt{1-t^2}Z
}
\tag{9.4}
$$

其中：

$$
Z\sim\mathcal N(0,I)
$$

注意时间方向：

$$
\tau:0\rightarrow\infty
$$

对应：

$$
t:1\rightarrow0
$$

因此随机插值从：

$$
t=0
$$

的高斯噪声出发，在：

$$
t=1
$$

达到真实数据。

这说明 VP-SDE 的边缘概率路径可以用 One-sided Interpolant 表示。

### 43. 为什么重新参数化后能避免某些奇异性？

如果直接将 OU-SDE 的时间变量：

$$
\tau
$$

改成：

$$
t=e^{-\tau}
$$

则：

$$
\frac{d\tau}{dt}=-\frac1t
$$

因此原 SDE 的漂移和扩散系数会出现：

$$
\frac1t
$$

或：

$$
\frac1{\sqrt t}
$$

这样的因子。

当：

$$
t\rightarrow0
$$

时，它们会发散。

但是随机插值并不要求保留原 OU-SDE 的动力学。

它只保留：

$$
p_t
$$

这条边缘概率路径。

然后重新构造：

$$
dY_t=
[b(t,Y_t)+\varepsilon(t)s(t,Y_t)]dt
+
\sqrt{2\varepsilon(t)}dW_t
$$

其中：

$$
\varepsilon(t)
$$

可以重新选择。

这样就可以避免直接时间变换带来的某些奇异扩散系数。

但需要注意，重新构造的漂移项仍然要检查端点极限，不能只因为选择了有限的 $\varepsilon$，就认为所有公式都自动数值稳定。

### 44. Denoising 为什么可以看成贝叶斯估计？

考虑 One-sided Interpolant：

$$
X_t=\alpha(t)Z+\beta(t)X_1
$$

希望根据：

$$
X_t=x
$$

估计真实数据：

$$
X_1
$$

在平方误差意义下，最优估计是：

$$
\boxed{
\hat X_1(t,x)
=
\mathbb E[X_1\mid X_t=x]
}
\tag{9.5}
$$

根据插值关系：

$$
X_1=\frac{X_t-\alpha Z}{\beta}
$$

在给定 $X_t=x$ 后取期望：

$$
\boxed{
\hat X_1(t,x)
=
\frac{x-\alpha(t)\eta_z(t,x)}{\beta(t)}
}
\tag{9.6}
$$

其中：

$$
\eta_z(t,x)=\mathbb E[Z\mid X_t=x]
$$

这就是噪声预测与 $x_0$ 预测之间的解析转换关系。

它说明：

**预测噪声、预测干净数据、预测速度场，并不是完全独立的建模任务。**

在给定插值形式后，它们之间可能存在解析变换。

### 45. 为什么多步 Denoising 可以恢复 ODE？

论文进一步研究：

如果每次根据当前带噪状态估计干净数据，然后重新组合下一个时刻的状态，是否能够得到一个连续时间生成过程？

考虑：

$$
X_t=\alpha(t)Z+\beta(t)X_1
$$

在时刻 $t$，估计：

$$
\eta_z(t,x)
$$

以及：

$$
\eta_1(t,x)
=
\frac{x-\alpha(t)\eta_z(t,x)}{\beta(t)}
$$

现在从 $t$ 更新到：

$$
t+\Delta t
$$

定义：

$$
X_{t+\Delta t}
=
\alpha(t+\Delta t)\eta_z(t,X_t)
+
\beta(t+\Delta t)\eta_1(t,X_t)
\tag{9.7}
$$

代入 $\eta_1$：

$$
\begin{aligned}
X_{t+\Delta t}
={}&
\alpha(t+\Delta t)\eta_z(t,X_t)\\
&+
\frac{\beta(t+\Delta t)}{\beta(t)}
\left[
X_t-\alpha(t)\eta_z(t,X_t)
\right]
\end{aligned}
$$

整理：

$$
\boxed{
\begin{aligned}
X_{t+\Delta t}
={}&
\frac{\beta(t+\Delta t)}{\beta(t)}X_t\\
&+
\left[
\alpha(t+\Delta t)
-
\frac{\alpha(t)\beta(t+\Delta t)}{\beta(t)}
\right]\eta_z(t,X_t)
\end{aligned}
}
\tag{9.8}
$$

现在使用一阶 Taylor 展开：

$$
\alpha(t+\Delta t)
=
\alpha(t)+\dot\alpha(t)\Delta t+O(\Delta t^2)
$$

$$
\beta(t+\Delta t)
=
\beta(t)+\dot\beta(t)\Delta t+O(\Delta t^2)
$$

因此：

$$
\frac{\beta(t+\Delta t)}{\beta(t)}
=
1+\frac{\dot\beta(t)}{\beta(t)}\Delta t
+
O(\Delta t^2)
$$

而：

$$
\begin{aligned}
&\alpha(t+\Delta t)
-
\frac{\alpha(t)\beta(t+\Delta t)}{\beta(t)}\\
&=
\left[
\dot\alpha(t)
-
\frac{\alpha(t)\dot\beta(t)}{\beta(t)}
\right]\Delta t
+
O(\Delta t^2)
\end{aligned}
$$

代回式 (9.8)：

$$
\begin{aligned}
X_{t+\Delta t}
={}&
X_t\\
&+
\left[
\frac{\dot\beta(t)}{\beta(t)}X_t
+
\left(
\dot\alpha(t)
-
\frac{\alpha(t)\dot\beta(t)}{\beta(t)}
\right)\eta_z(t,X_t)
\right]\Delta t\\
&+
O(\Delta t^2)
\end{aligned}
$$

两边减去 $X_t$，再除以 $\Delta t$：

$$
\frac{X_{t+\Delta t}-X_t}{\Delta t}
=
\frac{\dot\beta(t)}{\beta(t)}X_t
+
\left(
\dot\alpha(t)
-
\frac{\alpha(t)\dot\beta(t)}{\beta(t)}
\right)\eta_z(t,X_t)
+
O(\Delta t)
$$

令：

$$
\Delta t\rightarrow0
$$

得到：

$$
\boxed{
\frac{dX_t}{dt}
=
\frac{\dot\beta(t)}{\beta(t)}X_t
+
\left[
\dot\alpha(t)
-
\frac{\alpha(t)\dot\beta(t)}{\beta(t)}
\right]\eta_z(t,X_t)
}
\tag{9.9}
$$

观察发现，这正是前面推导的边缘速度场：

$$
b(t,x)
=
\frac{\dot\beta}{\beta}x
+
\left(
\dot\alpha-\frac{\alpha\dot\beta}{\beta}
\right)\eta_z
$$

因此：

$$
\boxed{
\frac{dX_t}{dt}=b(t,X_t)
}
$$

这说明：

**在步长趋于零、Denoiser 精确且满足适当正则性条件时，多步去噪更新可以收敛到对应的概率流 ODE。**

这是论文 Theorem 45 的核心结论。

它与我们之前讨论的 DDIM、DPM-Solver 和确定性去噪采样有很强的联系，但不能直接认为所有这些求解器都与式 (9.8) 完全相同。([arxiv.org](https://arxiv.org/html/2303.08797))

### 46. Rectified Flow 与 Reflow 的联系

论文第 5.3 节讨论了 Rectified Flow。

假设已经训练出一个理想速度场：

$$
b(t,x)
$$

并通过 ODE：

$$
\frac{dY_t}{dt}=b(t,Y_t)
$$

得到从初始位置到终点的映射：

$$
\boxed{
T(x_0)=Y_1(x_0)
}
\tag{9.10}
$$

其中：

$$
x_0\sim p_0
$$

理想情况下：

$$
T(X_0)\sim p_1
$$

现在构造新的端点耦合：

$$
(X_0,T(X_0))
$$

然后重新定义线性插值：

$$
\boxed{
\widetilde X_t=(1-t)X_0+tT(X_0)
}
\tag{9.11}
$$

新的瞬时速度：

$$
\dot{\widetilde X}_t=T(X_0)-X_0
$$

因此新的边缘速度为：

$$
\boxed{
\widetilde b(t,x)
=
\mathbb E[
T(X_0)-X_0
\mid
\widetilde X_t=x
]
}
\tag{9.12}
$$

这就是 Reflow 的基本构造。

如果定义：

$$
F_t(x_0)=(1-t)x_0+tT(x_0)
$$

并且 $F_t$ 对每个时刻都是可逆的，那么：

$$
x_0=F_t^{-1}(x)
$$

因此：

$$
\boxed{
\widetilde b(t,x)
=
T(F_t^{-1}(x))-F_t^{-1}(x)
}
\tag{9.13}
$$

沿着轨迹：

$$
\widetilde Y_t=F_t(x_0)
$$

有：

$$
\frac{d\widetilde Y_t}{dt}
=
T(x_0)-x_0
$$

这是常数。

所以：

$$
\boxed{
\widetilde Y_t=(1-t)x_0+tT(x_0)
}
\tag{9.14}
$$

也就是直线轨迹。

但是终点仍然是：

$$
\widetilde Y_1=T(x_0)
$$

因此：

$$
\boxed{
\widetilde Y_1=Y_1
}
\tag{9.15}
$$

在这个理想条件下，Reflow 改变了中间轨迹，却没有改变原来的终点映射。

### 47. 为什么 Reflow 可能提高采样效率？

原始 ODE：

$$
\dot Y_t=b(t,Y_t)
$$

可能具有复杂弯曲轨迹。

如果使用 Euler 积分：

$$
Y_{t+\Delta t}
=
Y_t+\Delta t\,b(t,Y_t)
$$

步长过大时会产生数值误差。

而理想 Reflow 后：

$$
\widetilde Y_t=(1-t)x_0+tT(x_0)
$$

速度恒定：

$$
\dot{\widetilde Y}_t=T(x_0)-x_0
$$

因此：

$$
\widetilde Y_1
=
x_0+[T(x_0)-x_0]
=
T(x_0)
$$

理论上一步 Euler 就能得到正确终点。

但这个结论依赖于理想速度场、正确的端点映射，以及重新插值映射的可逆性等条件。

有限模型训练误差可能导致 Reflow 后的终点分布发生偏移，因此不能把“理论上终点映射不变”直接理解为“实际重复训练绝不会引入误差”。

这与你之前做的 Reflow 实验完全对应。

## 第十部分：训练与采样的实际实现（论文第 6 节）

前面已经知道理论上需要学习：

$$
b(t,x)
$$

和：

$$
s(t,x)
$$

但实际训练时还需要考虑参数化、损失函数和端点数值稳定性。

### 48. 直接学习 $b$，还是分别学习 $v$ 和 $s$？

随机插值的瞬时速度：

$$
\dot X_t=\partial_tI+\dot\gamma Z
$$

边缘速度：

$$
b(t,x)
=
\mathbb E[\partial_tI\mid X_t=x]
+
\dot\gamma\mathbb E[Z\mid X_t=x]
$$

定义：

$$
v(t,x)=\mathbb E[\partial_tI\mid X_t=x]
$$

根据：

$$
\mathbb E[Z\mid X_t=x]=-\gamma s(t,x)
$$

得到：

$$
\boxed{
b(t,x)=v(t,x)-\gamma(t)\dot\gamma(t)s(t,x)
}
\tag{10.1}
$$

因此有两种训练方案。

**方案 A：直接学习完整边缘速度。**

$$
b_\theta(t,X_t)
\rightarrow
\partial_tI+\dot\gamma Z
$$

损失：

$$
\boxed{
\mathcal L_b
=
\mathbb E
\left[
\|b_\theta(t,X_t)-(\partial_tI+\dot\gamma Z)\|^2
\right]
}
\tag{10.2}
$$

**方案 B：分别学习插值速度和 Score。**

$$
v_\theta(t,X_t)\rightarrow\partial_tI
$$

$$
\eta_\theta(t,X_t)\rightarrow Z
$$

然后：

$$
\boxed{
b_\theta(t,x)
=
v_\theta(t,x)+\dot\gamma(t)\eta_\theta(t,x)
}
\tag{10.3}
$$

如果只需要 ODE，直接学习 $b$ 通常更简单。

如果还需要 SDE，则一般还需要 Score 或 Denoiser。

对于空间线性 One-sided Interpolant，只训练一个噪声预测网络就可以恢复两者。

### 49. 为什么直接训练 Score 可能数值不稳定？

Score 监督关系为：

$$
s(t,x)
=
-\frac1{\gamma(t)}
\mathbb E[Z\mid X_t=x]
$$

因此一种直接训练目标是：

$$
\mathcal L_s
=
\mathbb E
\left[
\left\|
s_\theta(t,X_t)+\frac{Z}{\gamma(t)}
\right\|^2
\right]
$$

但是：

$$
\gamma(0)=\gamma(1)=0
$$

所以当：

$$
t\rightarrow0,\quad t\rightarrow1
$$

时：

$$
\frac{Z}{\gamma(t)}
$$

可能非常大。

这会导致训练目标方差增加。

因此，论文建议优先考虑 Denoiser：

$$
\eta_\theta(t,X_t)\rightarrow Z
$$

它不需要在监督目标中除以：

$$
\gamma(t)
$$

所以更容易保持数值稳定。

### 50. Antithetic Sampling 为什么能降低方差？

论文还讨论了反向配对采样。

考虑：

$$
X_t^+=I(t,X_0,X_1)+\gamma(t)Z
$$

$$
X_t^-=I(t,X_0,X_1)-\gamma(t)Z
$$

两个样本使用相同的端点和相反的高斯噪声。

为什么这样有用？

考虑 Score 损失中的一项：

$$
\frac{Z^\top s_\theta(t,X_t)}{\gamma(t)}
$$

当 $\gamma(t)$ 很小时，对：

$$
s_\theta(t,I+\gamma Z)
$$

进行 Taylor 展开：

$$
s_\theta(t,I+\gamma Z)
=
s_\theta(t,I)
+
\gamma\nabla_xs_\theta(t,I)Z
+
O(\gamma^2)
$$

所以：

$$
\begin{aligned}
\frac{Z^\top s_\theta(t,I+\gamma Z)}{\gamma}
={}&
\frac{Z^\top s_\theta(t,I)}{\gamma}\\
&+
Z^\top\nabla_xs_\theta(t,I)Z
+
O(\gamma)
\end{aligned}
\tag{10.4}
$$

第一项：

$$
\frac{Z^\top s_\theta(t,I)}{\gamma}
$$

虽然对 $Z$ 的期望为零，但方差可能随着：

$$
\gamma\rightarrow0
$$

而发散。

现在使用正负噪声配对。

构造：

$$
\boxed{
A=
\frac{
Z^\top s_\theta(t,I+\gamma Z)
-
Z^\top s_\theta(t,I-\gamma Z)
}{
2\gamma
}
}
\tag{10.5}
$$

分别 Taylor 展开：

$$
s_\theta(t,I+\gamma Z)
=
s_\theta(t,I)+\gamma\nabla s_\theta Z+O(\gamma^2)
$$

$$
s_\theta(t,I-\gamma Z)
=
s_\theta(t,I)-\gamma\nabla s_\theta Z+O(\gamma^2)
$$

两式相减：

$$
s_\theta(t,I+\gamma Z)-s_\theta(t,I-\gamma Z)
=
2\gamma\nabla s_\theta Z+O(\gamma^3)
$$

代回：

$$
\boxed{
A=
Z^\top\nabla s_\theta(t,I)Z+O(\gamma^2)
}
\tag{10.6}
$$

原来可能发散的：

$$
\frac{Z^\top s_\theta(t,I)}{\gamma}
$$

被抵消了。

这就是 Antithetic Sampling 降低端点附近训练方差的数学原因。

### 51. 推理时如何选择 ODE 或 SDE？

训练完成后，如果得到：

$$
b_\theta(t,x)
$$

和：

$$
s_\theta(t,x)
$$

那么 ODE 为：

$$
\boxed{
dY_t=b_\theta(t,Y_t)dt
}
\tag{10.7}
$$

SDE 为：

$$
\boxed{
dY_t=
[b_\theta(t,Y_t)+\varepsilon(t)s_\theta(t,Y_t)]dt
+
\sqrt{2\varepsilon(t)}dW_t
}
\tag{10.8}
$$

使用 Euler–Maruyama 积分：

$$
\boxed{
Y_{k+1}
=
Y_k+
[b_\theta(t_k,Y_k)+\varepsilon(t_k)s_\theta(t_k,Y_k)]\Delta t
+
\sqrt{2\varepsilon(t_k)\Delta t}\,\xi_k
}
\tag{10.9}
$$

其中：

$$
\xi_k\sim\mathcal N(0,I)
$$

每一步重新采样：

$$
\xi_k
$$

这是 SDE 与原始随机插值的重要区别。

原始随机插值：

$$
X_t=I+\gamma Z
$$

中的 $Z$ 是整条插值轨迹固定的随机变量。

而 SDE 数值积分中的：

$$
\xi_0,\xi_1,\ldots
$$

是不同时间步的布朗增量。

### 52. 端点附近如何处理奇异性？

如果：

$$
s_\theta=-\frac{\eta_\theta}{\gamma}
$$

那么在：

$$
t=0,\quad t=1
$$

附近可能存在数值问题。

论文提出了几种处理方式：

第一种是让：

$$
\varepsilon(t)
$$

在端点附近趋于零，使 SDE 的 Score 修正项不会因为除以很小的 $\gamma$ 而失控。

第二种是提前停止积分：

$$
t_f<1
$$

然后使用一次 Denoising 更新到最终数据空间。

第三种是在 One-sided Interpolant 中利用解析关系消除某些表面上的奇异项。

例如：

$$
b(t,x)
=
\dot\alpha\eta_z+\dot\beta\eta_1
$$

即使：

$$
\frac{\dot\beta}{\beta}
$$

在：

$$
t=0
$$

形式上发散，原始条件期望表达式仍可能具有有限极限。

这说明：

**公式中出现分母趋于零，不一定意味着真实速度场发散；也可能只是参数化形式导致的可去奇异性。**

论文第 6 节对这些训练和采样策略进行了专门讨论。([arxiv.org](https://arxiv.org/html/2303.08797))

## 第十一部分：论文实验与结论（第 7～8 节）

### 53. 二维实验验证了什么？

论文首先使用二维目标分布，研究：

$$
\gamma(t)
$$

和：

$$
\varepsilon
$$

对生成效果的影响。

其中：

- $\gamma(t)$ 决定中间概率路径。
- $\varepsilon$ 决定生成 SDE 的随机性。

论文使用具有明显边界和复杂结构的二维目标密度，对比不同插值函数和扩散强度下的生成结果。

主要观察到：

**第一，适当的中间高斯噪声可以平滑概率路径。**

如果没有额外噪声：

$$
\gamma(t)=0
$$

中间概率密度可能产生复杂的结构。

增加适当噪声后，中间密度可能更加平滑，从而降低速度场学习难度。

**第二，适当的非零 SDE 扩散强度可以改善生成质量。**

在论文测试的二维配置中，使用：

$$
\varepsilon>0
$$

的 SDE 通常比相应 ODE 取得更好的密度估计结果。

但这不是普遍定理。

它取决于模型误差、Score 误差、扩散强度以及数值积分精度。

### 54. 128 维高斯混合分布实验

论文进一步使用 128 维高斯混合分布，比较不同学习方式。

主要包括：

$$
b
$$

直接速度预测；

$$
(v,s)
$$

分别学习插值速度和 Score；

以及噪声预测参数化：

$$
\eta_z
$$

实验发现，预测参数化会明显影响实际学习效果。

在所测试配置中，直接学习完整速度场通常具有较好的表现，而噪声预测相比直接 Score 预测也具有数值稳定性优势。

这说明虽然不同参数化在理论最优情况下可以互相转换，但有限模型容量、有限训练数据和优化误差会使它们产生不同的实际结果。

### 55. 图像生成实验

论文还将随机插值框架用于图像生成。

主要验证：

- One-sided Interpolant 能否用于高维图像数据。
- Mirror Interpolant 能否用于数据分布到自身的生成过程。
- ODE 和 SDE 是否都能作为实际采样方法。

这些实验主要展示框架的可行性，而不是证明随机插值在所有图像生成任务中优于其他方法。

论文完整实验设置及结果见第 7 节和附录 C。[arXiv](https://arxiv.org/html/2303.08797?utm_source=chatgpt.com)

---

# 第十二部分：整篇论文的理论体系总结

现在把所有内容统一起来。

## 56. 这篇论文真正解决了什么问题？

在传统生成模型中，我们往往先选择某种动力学。

例如：

**Flow Matching：**

$$
\frac{dX_t}{dt}=v_\theta(t,X_t)
$$

**Score-Based Diffusion：**

$$
dX_t=f_\theta(t,X_t)dt+g(t)dW_t
$$

然后研究如何训练这些动力学。

随机插值论文采用不同的思路：

**先规定概率路径，再构造能够实现这条路径的动力学。**

整个数学结构为：

$$
\boxed{
\begin{aligned}
&\text{选择端点分布 }p_0,p_1\\
&\Downarrow\\
&\text{选择端点耦合与插值函数}\\
&\Downarrow\\
&X_t=I(t,X_0,X_1)+\gamma(t)Z\\
&\Downarrow\\
&\text{得到边缘概率路径 }p_t\\
&\Downarrow\\
&\begin{cases}
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]\\
s(t,x)=\nabla\log p_t(x)
\end{cases}\\
&\Downarrow\\
&\begin{cases}
\text{ODE：}\quad dY_t=b(t,Y_t)dt\\[3pt]
\text{SDE：}\quad
dY_t=(b+\varepsilon s)dt+\sqrt{2\varepsilon}dW_t
\end{cases}
\end{aligned}
}
$$

在理想条件下，两种动力学具有相同的每时刻边缘分布。

## 57. 这篇论文最重要的理论贡献

我认为可以总结为五点。

**第一，概率路径与采样动力学解耦。**

随机插值先构造概率路径：

$$
p_t
$$

然后允许通过 ODE 或不同扩散强度的 SDE 实现它。

**第二，统一了速度场与 Score 的学习。**

速度场：

$$
b(t,x)=\mathbb E[\dot X_t\mid X_t=x]
$$

Score：

$$
s(t,x)=-\frac1{\gamma(t)}\mathbb E[Z\mid X_t=x]
$$

都可以通过条件期望回归学习。

**第三，解释了随机扩散为什么可能改善模型误差控制。**

对于具有正扩散强度的 SDE，可以通过漂移误差控制最终分布 KL。

而对于 ODE，论文给出的 KL 恒等式还涉及真实分布和模型分布之间的 Score 差异。

**第四，建立了与 Schrödinger Bridge 的联系。**

通过优化随机插值函数：

$$
I
$$

可以进一步优化中间概率路径，并在论文规定的条件下恢复 Schrödinger Bridge 的最优路径。

**第五，统一了多种生成模型的数学表达。**

One-sided Interpolant 可以联系 VP-SDE 和 CondOT Flow Matching。

Mirror Interpolant 可以描述数据分布到自身的过程。

Rectified Flow 则可以理解为对生成映射重新构造端点耦合与插值路径。

---

## 58. 对你研究 Flow Matching 轨迹规划最有价值的部分

结合你之前研究的扩散规划、CondOT、DPM-Solver 和强化学习，我认为最值得进一步研究的是：

### 方向一：将 Flow Matching ODE 转换为 SDE

假设当前规划模型学习：

$$
b_\theta(t,x,c)
$$

其中：

$$
c
$$

是场景条件。

如果额外学习：

$$
s_\theta(t,x,c)
$$

就可以构造：

$$
\boxed{
dX_t=
[b_\theta(t,X_t,c)+\varepsilon s_\theta(t,X_t,c)]dt
+
\sqrt{2\varepsilon}dW_t
}
\tag{12.1}
$$

这提供了在不重新设计目标概率路径的前提下，引入随机采样动力学的可能性。

但需要注意：

增加 SDE 随机性不必然增加最终轨迹分布的多模态程度。

因为在理想情况下，不同 $\varepsilon$ 具有相同的最终边缘分布。

它改变的是采样路径和端点耦合，而不是理论上的目标边缘分布。

因此，如果希望通过强化学习增加高奖励轨迹的概率，还需要额外的策略优化机制。

### 方向二：直接学习噪声，再恢复速度场

对于：

$$
X_t=\alpha(t)Z+\beta(t)X_1
$$

只需要训练：

$$
\eta_\theta(t,x,c)
$$

预测噪声。

然后：

$$
\boxed{
s_\theta(t,x,c)
=
-\frac{\eta_\theta(t,x,c)}{\alpha(t)}
}
$$

以及：

$$
\boxed{
b_\theta(t,x,c)
=
\frac{\dot\beta(t)}{\beta(t)}x
+
\left[
\dot\alpha(t)
-
\frac{\alpha(t)\dot\beta(t)}{\beta(t)}
\right]\eta_\theta(t,x,c)
}
$$

因此，一个网络可以同时用于 ODE 和 SDE 采样。

这与你之前研究的 $x_0$ 预测、速度预测和噪声预测参数化非常相关。

### 方向三：优化插值路径，而不仅仅优化求解器

你之前比较过：

- Euler。
- Midpoint。
- DPM-Solver。
- DPM-Solver++ 2M。
- Reflow。

这些方法主要关注如何更高效地求解给定动力学。

但随机插值论文还提出另一个思路：

**如果中间概率路径本身很复杂，那么即使使用高阶求解器，模型学习和数值积分也可能很困难。**

因此，可以同时研究：

$$
\boxed{
\text{概率路径设计}
+
\text{模型预测参数化}
+
\text{ODE/SDE 求解器}
}
$$

这三个维度的联合优化。

---

## 最后：哪些结论需要特别避免误解？

| 容易产生的理解 | 更准确的结论 |
|---|---|
| 随机插值就是 SDE | 随机插值本身通常具有可微样本路径，不等于布朗运动驱动的 SDE |
| ODE 与 SDE 完全等价 | 理想情况下可以具有相同的单时刻边缘分布，但轨迹和端点耦合不同 |
| $f=b+\varepsilon s$ 是唯一漂移 | 它是通过匹配概率流量构造的一个充分解 |
| 增加 $\varepsilon$ 就能增加最终多模态 | 理想情况下最终边缘分布不变 |
| SDE 一定比 ODE 更好 | 论文证明了不同的误差控制性质，但实际性能依赖训练与数值误差 |
| 任意随机插值都是 Schrödinger Bridge | 只有进一步优化概率路径并满足相关条件时，才能恢复最优 Schrödinger Bridge |
| Reflow 一定不会改变生成结果 | 理想条件下可以保持终点映射，实际训练误差可能引入偏移 |

**这篇论文最值得掌握的不是某一个特定的插值公式，而是以下思想：**

$$
\boxed{
\text{同一条概率路径，可以由不同的随机过程实现。}
}
$$

随机插值提供了一种直接构造概率路径的方法；条件期望提供了学习边缘速度和 Score 的方法；连续性方程和 Fokker–Planck 方程则建立了 ODE 与 SDE 之间的数学联系。

